<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/evaluacion_ragas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB2 · Evaluación RAGAS (Fase 1)

Evalúa las 171 respuestas generadas por NB1 (`respuestas_gpt4o.json`).

**Entorno AISLADO de NB1**: RAGAS 0.4 exige langchain-core 0.3.x, incompatible con el 1.x de NB1.

**Métricas**: Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecisionWithReference + la custom `deteccion_de_vigencia` (el corazón de la tesis).

**Orden**: correr celda por celda. La celda 5 tiene `MODO_PRUEBA=True` (6 respuestas). Verificar que sale bien y recién ahí `MODO_PRUEBA=False` para las 171.

## 1. Dependencias (pines exactos — no cambiar)

In [1]:
# ============================================================================
# NB2 · evaluacion_ragas.ipynb — CELDA 1: dependencias (entorno AISLADO)
# ============================================================================
# ⚠️ ESTE NOTEBOOK CORRE EN UN ENTORNO SEPARADO DE NB1.
#    RAGAS 0.4.3 exige langchain-core 0.3.x, que es INCOMPATIBLE con el
#    langchain-core 1.x que usa NB1 (LangGraph). Por eso son notebooks distintos:
#    NB1 genera los JSON, NB2 los evalúa. No se ejecutan juntos.
#
# ⚠️ pip install ragas "a secas" está ROTO (bug upstream: importa un módulo de
#    langchain-community que la 0.4 eliminó). Hay que PINEAR estas versiones exactas.
# ----------------------------------------------------------------------------
!pip install -q \
  "ragas==0.4.3" \
  "langchain-core==0.3.86" \
  "langchain==0.3.27" \
  "langchain-community==0.3.27" \
  "langchain-openai==0.3.35" \
  "openai>=1.0.0"

# verificación de que importa (si esto falla, revisar los pines de arriba)
import ragas
print("ragas", ragas.__version__, "— import OK")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently tak

## 2. Setup: Drive, API key, cargar los JSON de NB1

In [2]:
# ============================================================================
# CELDA 2: setup — Drive, API key, cargar los JSON de NB1
# ============================================================================
import os, json, getpass
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# API key de OpenAI (el juez de RAGAS es GPT-4o)
# Lee la key desde Colab Secrets (ícono de llave 🔑 en la barra izquierda).
# El secret debe llamarse OPENAI_API_KEY y tener "Acceso al notebook" activado.
import os
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("API key cargada desde Colab Secrets ✓")
    except Exception as e:
        # fallback: pedirla a mano si el secret no está o no tiene acceso
        import getpass
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
        print("API key ingresada a mano")

# --- rutas (misma estructura que NB1) ---
CARPETA = "/content/drive/MyDrive/tesis_chatbot/ragas"
PATH_RESP = os.path.join(CARPETA, "respuestas_gpt4o.json")

with open(PATH_RESP, encoding="utf-8") as f:
    data = json.load(f)

META = data["meta"]
RESP = data["respuestas"]
print("Cargado:", PATH_RESP)
print("Meta:", META)
print("Total respuestas:", len(RESP), "(esperado 171 = 57×3)")

# índice rápido por (id, brazo)
POR_CLAVE = {(r["id"], r["brazo"]): r for r in RESP}
BRAZOS = sorted({r["brazo"] for r in RESP})
IDS = sorted({r["id"] for r in RESP})
print("Brazos:", BRAZOS)
print("Preguntas:", len(IDS))


Mounted at /content/drive
API key cargada desde Colab Secrets ✓
Cargado: /content/drive/MyDrive/tesis_chatbot/ragas/respuestas_gpt4o.json
Meta: {'generado': '2026-07-18T18:13:28.285990', 'modo': 'completo', 'n_preguntas': 57, 'n_brazos': 3, 'top_k': 5, 'llm': 'gpt-4o'}
Total respuestas: 171 (esperado 171 = 57×3)
Brazos: ['A_baseline', 'B_graphrag', 'C_agente']
Preguntas: 57


## 3. Juez (GPT-4o) y las 4 métricas estándar

In [3]:
# ============================================================================
# CELDA 3: instanciar el JUEZ (LLM + embeddings) y las MÉTRICAS de RAGAS
# ============================================================================
# El juez es GPT-4o (mismo modelo que generó, pero como los 3 brazos usan el
# mismo generador, el sesgo de auto-preferencia es SIMÉTRICO y se cancela en la
# comparación entre brazos. Documentarlo en Limitaciones).
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasEmbeddings

_client = AsyncOpenAI()   # ASYNC: las métricas usan .ascore() (asíncrono)

# LLM juez y embeddings (para AnswerRelevancy)
judge_llm = llm_factory("gpt-4o", client=_client, max_tokens=4000)  # max_tokens alto: faithfulness descompone respuestas largas
judge_emb = RagasEmbeddings(client=_client, model="text-embedding-3-small")

# --- Las 4 métricas estándar (API collections, RAGAS 0.4) ---
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecisionWithReference
)
m_faith   = Faithfulness(llm=judge_llm)
m_relev   = AnswerRelevancy(llm=judge_llm, embeddings=judge_emb)
m_recall  = ContextRecall(llm=judge_llm)                 # necesita reference (gold)
m_precis  = ContextPrecisionWithReference(llm=judge_llm)  # necesita reference (gold)

print("Juez y métricas estándar listos.")
print("Firmas (recordatorio):")
print("  Faithfulness.ascore(user_input, response, retrieved_contexts)   -> sin gold")
print("  AnswerRelevancy.ascore(user_input, response)                    -> sin gold")
print("  ContextRecall.ascore(user_input, retrieved_contexts, reference) -> CON gold")
print("  ContextPrecisionWithReference.ascore(user_input, reference, retrieved_contexts) -> CON gold")


Juez y métricas estándar listos.
Firmas (recordatorio):
  Faithfulness.ascore(user_input, response, retrieved_contexts)   -> sin gold
  AnswerRelevancy.ascore(user_input, response)                    -> sin gold
  ContextRecall.ascore(user_input, retrieved_contexts, reference) -> CON gold
  ContextPrecisionWithReference.ascore(user_input, reference, retrieved_contexts) -> CON gold


## 4. Métrica custom: detección de vigencia

La única que detecta el fallo central del baseline (citar ley derogada como vigente). Usa el campo `rubrica` del set gold como criterio.

In [6]:
# ============================================================================
# CELDA 4 (v2): MÉTRICA CUSTOM · deteccion_de_vigencia  — PROMPT CORREGIDO
# ============================================================================
# FIX (v2): la v1 mezclaba "¿acierta la vigencia?" con "¿cumple TODA la rúbrica?"
# (incluidos detalles no-vigencia como fechas o artículos). Eso marcaba incorrecto
# respuestas que SÍ avisaban la derogación pero omitían un detalle menor → deprimía
# el resultado (5/17 falso en vez del real). Ahora la métrica juzga SOLO una cosa:
# ¿la respuesta presenta una norma derogada como vigente, o avisa correctamente?
from ragas.metrics import discrete_metric
from ragas.metrics.result import MetricResult
from openai import OpenAI

_c = OpenAI()

PROMPT_VIGENCIA = """Sos un evaluador jurídico. Tu ÚNICA tarea es determinar si la RESPUESTA
maneja correctamente el ESTADO DE VIGENCIA de las normas, comparando con los HECHOS.

NO evalúes completitud, ni estilo, ni si menciona todos los detalles (fechas exactas,
números de artículo, etc.). SOLO te importa la vigencia.

HECHOS DE REFERENCIA (la verdad sobre qué está vigente y qué fue derogado):
{gold}

RESPUESTA A EVALUAR:
{response}

Clasificá en UNA palabra:
- "correcto"   si la respuesta NO presenta como vigente ninguna norma derogada, y cuando
               corresponde, advierte la derogación o el reemplazo. (Aunque le falten detalles.)
- "incorrecto" si presenta una norma DEROGADA como si estuviera vigente, o afirma un estado
               de vigencia equivocado.
- "no_aplica"  si la pregunta/respuesta no involucra el estado de vigencia de ninguna norma.
Palabra:"""

@discrete_metric(name="deteccion_de_vigencia",
                 allowed_values=["correcto", "incorrecto", "no_aplica"])
def deteccion_de_vigencia(response: str, gold: str, rubrica: str = "") -> MetricResult:
    # sin gold no hay verdad de vigencia contra la cual comparar
    if not gold:
        return MetricResult(value="no_aplica", reason="sin gold de vigencia")
    prompt = PROMPT_VIGENCIA.format(gold=gold, response=response)
    out = _c.chat.completions.create(
        model="gpt-4o", temperature=0,
        messages=[{"role": "user", "content": prompt}],
    ).choices[0].message.content.strip().lower()
    if "incorrecto" in out:
        val = "incorrecto"
    elif "correcto" in out:
        val = "correcto"
    else:
        val = "no_aplica"
    return MetricResult(value=val, reason=out[:120])

# prueba en las 4 que antes daban incorrecto mal (deberían dar correcto ahora)
for pid in ["P1-06", "P1-14", "P2-12"]:
    _d = POR_CLAVE.get((pid, "B_graphrag"))
    if _d:
        r = deteccion_de_vigencia.score(response=_d["respuesta"], gold=_d["gold"], rubrica=_d.get("rubrica",""))
        print(f"{pid}/B_graphrag → {r.value}  ({r.reason[:50]})")


P1-06/B_graphrag → incorrecto  (incorrecto)
P1-14/B_graphrag → incorrecto  (incorrecto)
P2-12/B_graphrag → incorrecto  (incorrecto)


## 5. Ejecutar la evaluación (con guardado incremental)

Guarda tras **cada** respuesta. Si Colab se desconecta (pantalla apagada, timeout), volvé a correr esta misma celda: **retoma donde quedó**, no repite lo hecho.

⚠️ Empezá con `MODO_PRUEBA=True` (6). Si sale bien, `MODO_PRUEBA=False` (171). La corrida completa puede tardar 30-60 min.

In [5]:
# ============================================================================
# CELDA 5: EJECUTAR la evaluación — CON GUARDADO INCREMENTAL Y RESUME
# ============================================================================
# Guarda después de CADA respuesta. Si el runtime se desconecta (pantalla
# apagada, timeout de Colab), volvés a correr esta MISMA celda y RETOMA desde
# donde quedó — no repite lo ya hecho ni re-paga las llamadas.
#
# Reglas especiales (§16.5): fuera_alcance se EXCLUYE de AnswerRelevancy
# (castiga fallback honesto) → flag fallback_ok; gold vacío salta recall/precision.
import asyncio, time, json, os
from datetime import datetime

MODO_PRUEBA = False     # <-- False = las 171. True = 6 de prueba.
N_PRUEBA = 6
PAUSA = 1.5

sufijo = "_PRUEBA" if MODO_PRUEBA else ""
OUT_PATH = os.path.join(CARPETA, f"metricas_ragas{sufijo}.json")

# --- RESUME: cargar lo ya hecho (si existe) y saltearlo ---
resultados = []
hechas = set()
if os.path.exists(OUT_PATH):
    try:
        prev = json.load(open(OUT_PATH, encoding="utf-8"))
        todas_prev = prev.get("resultados", [])
        # una fila está COMPLETA si no tiene error ni métricas falladas
        def _completa(r):
            return ("error" not in r) and not any(k.startswith("err_") for k in r)
        resultados = [r for r in todas_prev if _completa(r)]   # conservar solo las buenas
        hechas = {(r["id"], r["brazo"]) for r in resultados}
        n_fallidas = len(todas_prev) - len(resultados)
        print(f"↻ RESUME: {len(hechas)} completas se saltean; {n_fallidas} incompletas se reintentan.")
    except Exception as e:
        print("No se pudo leer parcial, empiezo de cero:", e)

def es_fuera_alcance(r):  return "fuera_alcance" in (r.get("tipo") or [])
def texto_pregunta(r):    return r.get("pregunta_usada") or r.get("pregunta_original")

def guardar():
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump({"meta": {"generado": datetime.now().isoformat(),
                            "n": len(resultados), "completo": None},
                   "resultados": resultados}, f, ensure_ascii=False, indent=2)

async def evaluar_una(r):
    q, resp, ctxs = texto_pregunta(r), r["respuesta"], r["contextos"]
    gold, rub = r.get("gold") or "", r.get("rubrica") or ""
    fila = {"id": r["id"], "brazo": r["brazo"],
            "afectada_por_reforma": r.get("afectada_por_reforma", False),
            "tipo": r.get("tipo") or []}
    try: fila["faithfulness"] = (await m_faith.ascore(user_input=q, response=resp, retrieved_contexts=ctxs)).value
    except Exception as e: fila["faithfulness"]=None; fila["err_faith"]=str(e)[:80]
    if es_fuera_alcance(r):
        fila["answer_relevancy"]=None
        fila["fallback_ok"]=any(k in resp.lower() for k in
            ["no tengo","no cuento","no dispongo","no puedo","fuera de","no encontré","no hay información"])
    else:
        try: fila["answer_relevancy"]=(await m_relev.ascore(user_input=q, response=resp)).value
        except Exception as e: fila["answer_relevancy"]=None; fila["err_relev"]=str(e)[:80]
    if gold:
        try: fila["context_recall"]=(await m_recall.ascore(user_input=q, retrieved_contexts=ctxs, reference=gold)).value
        except Exception as e: fila["context_recall"]=None; fila["err_recall"]=str(e)[:80]
        try: fila["context_precision"]=(await m_precis.ascore(user_input=q, reference=gold, retrieved_contexts=ctxs)).value
        except Exception as e: fila["context_precision"]=None; fila["err_precis"]=str(e)[:80]
    else:
        fila["context_recall"]=None; fila["context_precision"]=None
    try: fila["deteccion_vigencia"]=deteccion_de_vigencia.score(response=resp, gold=gold, rubrica=rub).value
    except Exception as e: fila["deteccion_vigencia"]=None; fila["err_vig"]=str(e)[:80]
    return fila

async def correr(lista):
    for i, r in enumerate(lista, 1):
        if (r["id"], r["brazo"]) in hechas:
            continue   # ya evaluada en una corrida anterior
        for intento in range(4):
            try:
                fila = await evaluar_una(r)
                resultados.append(fila)
                hechas.add((r["id"], r["brazo"]))
                guardar()               # <-- GUARDADO INCREMENTAL tras cada una
                break
            except Exception as e:
                if "rate" in str(e).lower() and intento < 3:
                    espera = 2**intento; print(f"  429, esperando {espera}s..."); time.sleep(espera)
                else:
                    resultados.append({"id": r["id"], "brazo": r["brazo"], "error": str(e)[:100]})
                    hechas.add((r["id"], r["brazo"])); guardar(); break
        print(f"[{i}/{len(lista)}] {r['id']}/{r['brazo']}")
        time.sleep(PAUSA)

lote = RESP[:N_PRUEBA] if MODO_PRUEBA else RESP
faltan = [r for r in lote if (r["id"], r["brazo"]) not in hechas]
print(f"{'PRUEBA' if MODO_PRUEBA else 'COMPLETO'}: {len(lote)} totales, {len(faltan)} pendientes, {len(hechas)} ya hechas")

await correr(lote)

# marcar completo
prev = json.load(open(OUT_PATH, encoding="utf-8"))
prev["meta"]["completo"] = (len([r for r in resultados if "error" not in r]) >= len(lote))
json.dump(prev, open(OUT_PATH,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"\n✓ Guardado final: {OUT_PATH} ({len(resultados)} filas)")


↻ RESUME: 171 completas se saltean; 0 incompletas se reintentan.
COMPLETO: 171 totales, 0 pendientes, 171 ya hechas

✓ Guardado final: /content/drive/MyDrive/tesis_chatbot/ragas/metricas_ragas.json (171 filas)


## 6. Análisis — la tabla de la tesis

Promedios por brazo + el titular: detección de vigencia en las 17 preguntas de la reforma.

In [10]:
# ============================================================================
# CELDA 6: ANÁLISIS — la tabla que va a la tesis
# ============================================================================
# Promedios por brazo de cada métrica, más el desglose de deteccion_de_vigencia
# sobre las preguntas afectadas por la reforma (el titular de la tesis).
import pandas as pd

df = pd.DataFrame(resultados)
if "error" in df.columns:
    err = df[df["error"].notna()]
    if len(err): print("⚠ Filas con error:", len(err)); display(err[["id","brazo","error"]])

MET_NUM = ["faithfulness","answer_relevancy","context_recall","context_precision"]

# --- 1) Promedios por brazo (métricas numéricas) ---
print("="*60); print("PROMEDIOS POR BRAZO (métricas RAGAS estándar)"); print("="*60)
tabla = df.groupby("brazo")[MET_NUM].mean(numeric_only=True).round(3)
display(tabla)

# --- 2) deteccion_de_vigencia: conteo por brazo ---
print("\n"+"="*60); print("DETECCIÓN DE VIGENCIA — conteo por brazo"); print("="*60)
piv = df.pivot_table(index="brazo", columns="deteccion_vigencia",
                     values="id", aggfunc="count", fill_value=0)
display(piv)

# --- 3) EL TITULAR: vigencia SOLO en las afectadas por la reforma ---
print("\n"+"="*60); print("★ TITULAR: vigencia en las preguntas de la REFORMA"); print("="*60)
ref = df[df["afectada_por_reforma"] == True]
if len(ref):
    piv_ref = ref.pivot_table(index="brazo", columns="deteccion_vigencia",
                              values="id", aggfunc="count", fill_value=0)
    display(piv_ref)
    # tasa de acierto por brazo
    print("\nTasa de 'correcto' sobre afectadas por reforma:")
    for b in sorted(ref["brazo"].unique()):
        sub = ref[ref["brazo"] == b]
        ok = (sub["deteccion_vigencia"] == "correcto").sum()
        print(f"  {b}: {ok}/{len(sub)}")

# --- 4) fallback en fuera_alcance ---
if "fallback_ok" in df.columns:
    print("\n"+"="*60); print("FALLBACK HONESTO (fuera_alcance)"); print("="*60)
    fa = df[df["fallback_ok"].notna()]
    for b in sorted(fa["brazo"].unique()):
        sub = fa[fa["brazo"] == b]
        ok = sub["fallback_ok"].sum()
        print(f"  {b}: {ok}/{len(sub)} admitió no saber")

# --- 5) guardar la tabla resumen ---
tabla.to_csv(os.path.join(CARPETA, "resumen_metricas.csv"))
print("\n✓ Resumen guardado en resumen_metricas.csv")


PROMEDIOS POR BRAZO (métricas RAGAS estándar)


,faithfulness,answer_relevancy,context_recall,context_precision
brazo,,,,
A_baseline,0.955,0.630,0.262,0.335
B_graphrag,0.942,0.673,0.513,0.450
C_agente,0.784,0.569,0.571,0.376



DETECCIÓN DE VIGENCIA — conteo por brazo


deteccion_vigencia,correcto,incorrecto,no_aplica
brazo,,,
A_baseline,3,20,34
B_graphrag,11,12,34
C_agente,9,15,33



★ TITULAR: vigencia en las preguntas de la REFORMA


deteccion_vigencia,correcto,incorrecto,no_aplica
brazo,,,
A_baseline,2,13,2
B_graphrag,5,10,2
C_agente,3,12,2



Tasa de 'correcto' sobre afectadas por reforma:
  A_baseline: 2/17
  B_graphrag: 5/17
  C_agente: 3/17

FALLBACK HONESTO (fuera_alcance)
  A_baseline: 2/10 admitió no saber
  B_graphrag: 2/10 admitió no saber
  C_agente: 8/10 admitió no saber

✓ Resumen guardado en resumen_metricas.csv


## 7. (Opcional) Re-evaluar solo la vigencia

Si cambiás el prompt de `deteccion_de_vigencia` (celda 4), corré ESTA celda para recalcular solo esa columna sobre las 171 — sin re-pagar las 4 métricas RAGAS. Después volvé a correr la celda 6.

In [5]:
# ============================================================================
# CELDA 7 (opcional): RE-EVALUAR solo deteccion_de_vigencia sobre las 171
# ============================================================================
# Se usa cuando cambia el prompt de la métrica de vigencia (v2) y NO querés
# re-pagar las 4 métricas RAGAS (que no cambiaron). Lee metricas_ragas.json,
# recalcula SOLO la columna deteccion_vigencia usando las respuestas de NB1,
# y regraba. Barato: 171 llamadas de 1 token de salida.
import json, os, time

RESP_PATH = os.path.join(CARPETA, "respuestas_gpt4o.json")
MET_PATH  = os.path.join(CARPETA, "metricas_ragas.json")

respuestas = {(r["id"], r["brazo"]): r for r in json.load(open(RESP_PATH, encoding="utf-8"))["respuestas"]}
met = json.load(open(MET_PATH, encoding="utf-8"))

n=0
for fila in met["resultados"]:
    clave = (fila["id"], fila["brazo"])
    src = respuestas.get(clave)
    if not src:
        continue
    nueva = deteccion_de_vigencia.score(response=src["respuesta"],
                                        gold=src.get("gold",""), rubrica=src.get("rubrica",""))
    fila["deteccion_vigencia"] = nueva.value
    n += 1
    if n % 30 == 0: print(f"  {n}/{len(met['resultados'])}")
    time.sleep(0.4)

json.dump(met, open(MET_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"✓ Re-evaluada deteccion_vigencia en {n} filas. Guardado.")
print("→ Ahora volvé a correr la CELDA 6 (análisis) para ver la tabla actualizada.")


  30/171
  60/171
  90/171
  120/171
  150/171
✓ Re-evaluada deteccion_vigencia en 171 filas. Guardado.
→ Ahora volvé a correr la CELDA 6 (análisis) para ver la tabla actualizada.


## Notas

- El juez GPT-4o evalúa respuestas de GPT-4o: sesgo de auto-preferencia SIMÉTRICO entre brazos (se cancela en la comparación). Documentar en Limitaciones.
- `deteccion_de_vigencia` sobre las afectadas por reforma = el resultado central.
- Salidas en Drive: `metricas_ragas.json`, `resumen_metricas.csv`.